In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, when, try_element_at, lit

In [ ]:
spark = SparkSession.builder \
		.appName('Ecommerce-Analytics-Silver-Normalizations') \
		.master('local[*]') \
		.config('spark.driver.memory', '6g') \
		.config('spark.driver.executor', '6g') \
		.config('spark.shuffle.partitions', '200') \
		.getOrCreate()

bronze_path = '../data/bronze/'
silver_path = '../data/silver/'

In [ ]:
df = spark.read.parquet(bronze_path)

In [ ]:
user_df = df.select('user_id').distinct().filter(col('user_id').isNotNull())
category_parsed = df.select('category_id', 'category_code').distinct() \
        .filter(col('category_id').isNotNull()) \
        .withColumn('code_parts', split(col('category_code'), '\.'))

<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_6117/2220561384.py:4: SyntaxWarning: invalid escape sequence '\.'
  .withColumn('code_parts', split(col('category_code'), '\.'))


In [ ]:
print(category_parsed)

DataFrame[category_id: bigint, category_code: string, code_parts: array<string>]


In [ ]:
category_parsed.select('code_parts').show(5)

+--------------------+
|          code_parts|
+--------------------+
|[appliances, sewi...|
|[electronics, cam...|
|[appliances, kitc...|
|  [apparel, costume]|
|[apparel, underwear]|
+--------------------+
only showing top 5 rows


In [ ]:
categories_df = category_parsed.withColumn(
    "main_category", try_element_at(col("code_parts"), lit(1))
).withColumn(
    "sub_category", try_element_at(col("code_parts"), lit(2))
).withColumn(
    "product_type", try_element_at(col("code_parts"), lit(3))
).drop("code_parts")

In [ ]:
category_df.show(2)

26/06/01 15:53:39 ERROR Executor: Exception in task 0.0 in stage 28.0 (TID 226)]
org.apache.spark.SparkArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX_IN_ELEMENT_AT] The index 3 is out of bounds. The array has 2 elements. Use `try_element_at` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003
== DataFrame ==
"element_at" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidElementAtIndexError(QueryExecutionErrors.scala:256)
	at org.apache.spark.sql.errors.QueryExecutionErrors.invalidElementAtIndexError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.hashAgg_doAggregateWithKeysOutput_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowI

ArrayIndexOutOfBoundsException: [INVALID_ARRAY_INDEX_IN_ELEMENT_AT] The index 3 is out of bounds. The array has 2 elements. Use `try_element_at` to tolerate accessing element at invalid index and return NULL instead. SQLSTATE: 22003
== DataFrame ==
"element_at" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
